# Multiplicative miscalibration of binary logistic regression models: figures

This notebook recreates the eleven figures of the paper. It runs in an **R kernel** (IRkernel).

* Figures 1-3 use the iris data that come with R.
* Figures 4-6 are simulations; their seeds match the paper.
* Figures 7-11 use speaker embeddings from `data/ecapa_embeddings.csv`. The embeddings were computed with the ECAPA-TDNN network from 400 ms clips of the recordings; the recordings themselves are not distributed.

Required R packages: `logistf`, `arm`, `glmnet`, `MASS`. The whole notebook runs in under a minute.

In [ ]:
for (p in c("logistf", "arm", "glmnet", "MASS"))
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p)


## Part 1: iris data

### Figure 1: Normal Q-Q plots

Normal Q-Q plots of petal width for versicolor and virginica. Points close to the reference line indicate that the class-conditional distribution is approximately normal.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 3.2)

d_qq <- droplevels(subset(iris, Species %in% c("versicolor", "virginica")))

col_versicolor <- "#56B4E9"   # sky blue (species colours differ from model colours)
col_virginica  <- "#CC79A7"   # reddish purple

par(mfrow = c(1, 2), mar = c(4, 4.5, 3, 1))

pw_versicolor <- d_qq$Petal.Width[d_qq$Species == "versicolor"]
qqnorm(pw_versicolor, col = col_versicolor, pch = 19,
       main = "versicolor", xlab = "Theoretical quantiles",
       ylab = "petal width (cm)")
qqline(pw_versicolor, col = col_versicolor, lwd = 2)

pw_virginica <- d_qq$Petal.Width[d_qq$Species == "virginica"]
qqnorm(pw_virginica, col = col_virginica, pch = 19,
       main = "virginica", xlab = "Theoretical quantiles",
       ylab = "petal width (cm)")
qqline(pw_virginica, col = col_virginica, lwd = 2)



### Figure 2: LDA, QDA and logistic regression on iris

Comparison of LDA, QDA, and logistic regression on discrimination of *Iris virginica* from *Iris versicolor*.

In [ ]:
options(repr.plot.width = 4.3, repr.plot.height = 2.7)

suppressPackageStartupMessages(library(MASS))

d <- droplevels(subset(iris, Species %in% c("versicolor", "virginica")))
d$y <- as.integer(d$Species == "virginica")

lda_fit <- lda(Species ~ Petal.Width, data = d)
qda_fit <- qda(Species ~ Petal.Width, data = d)
glm_fit <- glm(y ~ Petal.Width, data = d, family = binomial)

xg <- seq(min(d$Petal.Width), max(d$Petal.Width), length.out = 400)
nd <- data.frame(Petal.Width = xg)

p_lda <- predict(lda_fit, newdata = nd)$posterior[, "virginica"]
p_qda <- predict(qda_fit, newdata = nd)$posterior[, "virginica"]
p_glm <- predict(glm_fit, newdata = nd, type = "response")

col_lda <- "#1b9e77"
col_glm <- "#d95f02"
col_qda <- "#7570b3"

par(mar = c(3.4, 4, 1.8, 0.5), mgp = c(2.2, 0.7, 0), cex.lab = 0.85, cex.axis = 0.85, cex.main = 0.95)
plot(xg, p_lda, type = "l", lwd = 2, col = col_lda, yaxt = "n",
     xlab = "petal width (cm)", ylab = "P(virginica | petal width)",
     main = "Predicted probability curves", ylim = c(0, 1))
lines(xg, p_glm, lwd = 2, lty = 2, col = col_glm)
lines(xg, p_qda, lwd = 2, lty = 4, col = col_qda)

## y-axis on [0,1], tick labels to 1 decimal place
y_ticks <- seq(0, 1, by = 0.2)
axis(2, at = y_ticks, labels = sprintf("%.1f", y_ticks), las = 1)

legend("bottomright", bty = "n", cex = 0.8, legend = c("LDA", "Logistic regression", "QDA"),
       col = c(col_lda, col_glm, col_qda), lwd = 2, lty = c(1, 2, 4))

### Figure 3: Model-based reliability diagram

Reliability of LDA and logistic regression against QDA as the reference, for discrimination of *Iris virginica* from *Iris versicolor*. Top: predicted probability of virginica versus the QDA probability; the dotted diagonal marks perfect agreement, and the arrows mark each model's largest departure. Bottom: number of the 100 observed flowers of each species in each bin (width 0.05) of the probability predicted by logistic regression.

In [ ]:
options(repr.plot.width = 4.3, repr.plot.height = 3.9)

suppressPackageStartupMessages(library(MASS))

d_rel <- droplevels(subset(iris, Species %in% c("versicolor", "virginica")))
d_rel$y <- as.integer(d_rel$Species == "virginica")

lda_rel <- lda(Species ~ Petal.Width, data = d_rel)
qda_rel <- qda(Species ~ Petal.Width, data = d_rel)
glm_rel <- glm(y ~ Petal.Width, data = d_rel, family = binomial)

predict_rel <- function(nd) list(
  lda = predict(lda_rel, newdata = nd)$posterior[, "virginica"],
  qda = predict(qda_rel, newdata = nd)$posterior[, "virginica"],
  glm = unname(predict(glm_rel, newdata = nd, type = "response")))

## Curves over a fine grid of petal widths; predictions at the observed flowers.
xg_rel <- seq(min(d_rel$Petal.Width), max(d_rel$Petal.Width), length.out = 400)
pg <- predict_rel(data.frame(Petal.Width = xg_rel))
pd <- predict_rel(data.frame(Petal.Width = d_rel$Petal.Width))

col_lda <- "#1b9e77"   # as in Figure 2
col_glm <- "#d95f02"

layout(matrix(1:2, ncol = 1), heights = c(3.4, 1.4))

## Top: reliability-style curves against the QDA reference.
par(mar = c(0.6, 4.6, 1.6, 0.8), mgp = c(2.8, 0.7, 0), cex = 0.85)
plot(NA, xlim = c(0, 1), ylim = c(0, 1), xaxt = "n", las = 1, xaxs = "i", yaxs = "i",
     xlab = "", ylab = "QDA probability (reference)",
     main = "Reliability against QDA", cex.main = 1)
abline(0, 1, col = "grey50", lty = 3)
lines(pg$lda, pg$qda, lwd = 2, col = col_lda)
lines(pg$glm, pg$qda, lwd = 2, lty = 2, col = col_glm)

## Maximal departure of each model: vertical double-headed arrow from the
## diagonal (perfect agreement) to the curve, labeled with its size.
mark_max <- function(p, col, side, at = 0.5) {
  i <- which.max(abs(p - pg$qda))
  x0 <- p[i]; y_diag <- p[i]; y_curve <- pg$qda[i]
  arrows(x0, y_diag, x0, y_curve, code = 3, angle = 25, length = 0.06,
         lwd = 1.6, col = col)
  text(x0, y_diag + at * (y_curve - y_diag), sprintf("%.2f", abs(y_curve - y_diag)),
       pos = side, col = col, cex = 0.9, font = 2)
}
mark_max(pg$lda, col_lda, side = 2, at = 1.05)
mark_max(pg$glm, col_glm, side = 4)

legend("topleft", bty = "n", legend = c("LDA", "Logistic regression", "Perfect agreement"),
       col = c(col_lda, col_glm, "grey50"), lwd = c(2, 2, 1), lty = c(1, 2, 3), cex = 0.9)
legend("bottomright", bty = "n", title = "Flowers (bottom panel)", cex = 0.85,
       legend = c("versicolor", "virginica"), fill = c("#56B4E9", "#CC79A7"), border = NA)

## Bottom: number of flowers in each bin of the logistic-regression predicted
## probability, one bar per species side by side (species colours as in Figure 1).
par(mar = c(3.6, 4.6, 0.4, 0.8), mgp = c(2.4, 0.7, 0))
col_sp <- c(versicolor = "#56B4E9", virginica = "#CC79A7")
sp <- as.character(d_rel$Species)
br <- seq(0, 1, by = 0.05)
mids <- head(br, -1) + 0.025
bw <- 0.018
h_ver <- hist(pd$glm[sp == "versicolor"], breaks = br, plot = FALSE)$counts
h_vir <- hist(pd$glm[sp == "virginica"],  breaks = br, plot = FALSE)$counts
plot(NA, xlim = c(0, 1), ylim = c(0, max(h_ver, h_vir) * 1.1), xaxs = "i", las = 1,
     xlab = "Predicted probability of virginica (logistic regression)", ylab = "Flowers")
rect(mids - bw, 0, mids, h_ver, col = col_sp["versicolor"], border = NA)
rect(mids, 0, mids + bw, h_vir, col = col_sp["virginica"], border = NA)


## Part 2: simulated data

### Figure 4: Predictions on one simulated dataset

Predicted P(class B | x) from each of the five models, together with the true Bayes-optimal probability, for a single simulated dataset (the first of the Monte Carlo replications used in @fig-calibration-ratios below). Tick marks along the bottom show the sampled class A (dark) and class B (light) points.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.8)

suppressPackageStartupMessages({
  library(logistf)
  library(arm)
})

set.seed(1)   # same seed as the Monte Carlo simulation below -> its first replication
n   <- 50
mu1 <- 0; mu2 <- 6; sdv <- 1

x1 <- rnorm(n, mu1, sdv)
x2 <- rnorm(n, mu2, sdv)
x  <- c(x1, x2)
y  <- factor(rep(c("A", "B"), each = n))
yb <- as.integer(y == "B")

xg <- seq(min(x) - 1, max(x) + 1, length.out = 400)

## True Bayes-optimal probability curve (same closed form as in the
## Monte Carlo simulation below).
b_true <- (mu2 - mu1) / sdv^2
a_true <- -(mu2^2 - mu1^2) / (2 * sdv^2)
p_true <- plogis(a_true + b_true * xg)

## Firth logistic regression.
firth_fit <- logistf(yb ~ x, pl = FALSE)
p_firth <- plogis(coef(firth_fit)[1] + coef(firth_fit)[2] * xg)

## Platt scaling.
Np <- sum(yb == 1); Nn <- sum(yb == 0)
t_std <- ifelse(yb == 1, (Np + 1) / (Np + 2), 1 / (Nn + 2))
p_platt  <- predict(suppressWarnings(glm(t_std ~ x, family = binomial)),
                     newdata = data.frame(x = xg), type = "response")

## LDA (pooled-variance closed form, as in the Monte Carlo simulation).
m1 <- mean(x1); m2 <- mean(x2)
s2_pooled <- ((n - 1) * var(x1) + (n - 1) * var(x2)) / (2 * n - 2)
b_lda <- (m2 - m1) / s2_pooled
a_lda <- -(m2^2 - m1^2) / (2 * s2_pooled)
p_lda <- plogis(a_lda + b_lda * xg)

## Bayesian logistic regression, posterior mode via arm::bayesglm, with
## Gelman et al.'s (2008) data-independent weakly informative priors:
## Cauchy(0, 2.5) (prior.df = 1), and a Student-t (df = 7) variant.
cauchy_fit <- bayesglm(yb ~ x, family = binomial,
                       prior.scale = 2.5, prior.df = 1,
                       prior.scale.for.intercept = 10,
                       prior.df.for.intercept = 1)
p_cauchy <- plogis(coef(cauchy_fit)[1] + coef(cauchy_fit)[2] * xg)

t_fit <- bayesglm(yb ~ x, family = binomial,
                  prior.scale = 2.5, prior.df = 7,
                  prior.scale.for.intercept = 10,
                  prior.df.for.intercept = 7)
p_t <- plogis(coef(t_fit)[1] + coef(t_fit)[2] * xg)

curves <- list(Firth = p_firth,
               Platt = p_platt, LDA = p_lda,
               Cauchy = p_cauchy, StudentT = p_t)
cols <- c(Firth = "#7570b3",
          Platt = "#e7298a", LDA = "#e6ab02",
          Cauchy = "#a6761d", StudentT = "#666666")
ltys <- c(Firth = 4,
          Platt = 5, LDA = 2,
          Cauchy = 3, StudentT = 4)
labs <- c(Firth = "Firth logistic",
          Platt = "Platt scaling", LDA = "LDA",
          Cauchy = "Cauchy(0, 2.5) prior", StudentT = "Student-t(7) prior")

par(mar = c(4, 4.5, 1, 1))
plot(xg, p_true, type = "l", lwd = 3, col = "black",
     xlab = "x", ylab = "P(class B | x)", ylim = c(0, 1))
abline(h = 0.5, col = "grey50", lwd = 1, lty = 2)
for (mdl in names(curves)) {
  lines(xg, curves[[mdl]], lwd = 2, col = cols[mdl], lty = ltys[mdl])
}
rug(x1, col = "grey30")
rug(x2, col = "grey60")

## Nudge the legend up by half a text line from its default "bottomright" spot.
leg_probe <- legend("bottomright", bty = "n", cex = 0.85,
       legend = c("True (Bayes)", unname(labs[names(curves)])),
       col = c("black", unname(cols[names(curves)])),
       lwd = c(3, rep(2, length(curves))),
       lty = c(1, unname(ltys[names(curves)])),
       plot = FALSE)
line_h <- strheight("A", cex = 0.85)
legend(x = leg_probe$rect$left, y = leg_probe$rect$top + 0.5 * line_h,
       bty = "n", cex = 0.85,
       legend = c("True (Bayes)", unname(labs[names(curves)])),
       col = c("black", unname(cols[names(curves)])),
       lwd = c(3, rep(2, length(curves))),
       lty = c(1, unname(ltys[names(curves)])))

## Inset: class-conditional normal densities, upper-left quadrant.
col_A <- "grey30"
col_B <- "grey60"

d_true_A <- dnorm(xg, mu1, sdv)
d_true_B <- dnorm(xg, mu2, sdv)

peak_h <- max(d_true_A, d_true_B)

ip <- par(fig = c(0.13, 0.52, 0.50, 0.95), new = TRUE,
          mar = c(2.2, 2.5, 1, 0.5), cex = 0.7)
## Opaque background for the whole inset panel (plot box + margins), so
## whatever's behind it in the main plot (e.g. the P = 0.5 line) can't show
## through the inset's axis-label area.
plot.new()
rect(0, 0, 1, 1, col = "white", border = NA, xpd = TRUE)
par(new = TRUE)
plot(xg, d_true_A, type = "l", lwd = 1.5, col = col_A,
     xlab = "", ylab = "", ylim = c(0, peak_h * 1.3),
     xaxt = "n", yaxt = "n")
lines(xg, d_true_B, lwd = 1.5, col = col_B)
axis(1, tck = -0.05, mgp = c(0, 0.3, 0))
axis(2, tck = -0.05, mgp = c(0, 0.4, 0), las = 1)
title(xlab = "x", ylab = "density", mgp = c(1.1, 0, 0))
box()
text(mu1, dnorm(mu1, mu1, sdv), "class A", col = col_A, cex = 0.75, pos = 3)
text(mu2, dnorm(mu2, mu2, sdv), "class B", col = col_B, cex = 0.75, pos = 3)
par(ip)

### Monte Carlo simulation for Figure 5

Each of the 30 replications draws 50 points per class from N(0, 1) and N(6, 1) and fits the models. 
The cross-validated L1 and L2 fits are kept so that the random-number stream matches the paper; as explained in Section 4, they are not shown, because under separation cross-validation selects the end of the penalty grid. Runs in about a minute.

In [ ]:
suppressPackageStartupMessages({
  library(glmnet)   # L1/L2-penalized logistic regression
  library(logistf)  # Firth's penalized-likelihood logistic regression
  library(arm)      # bayesglm: Gelman et al. weakly informative priors
})

set.seed(1)

n_reps <- 30    # Monte Carlo replications (M = 30)
n      <- 50    # points per class
mu1    <- 0
mu2    <- 6
sdv    <- 1
ln10   <- log(10)

fit_one_rep <- function(rep_id) {
  x1 <- rnorm(n, mu1, sdv)
  x2 <- rnorm(n, mu2, sdv)
  x  <- c(x1, x2)
  y  <- factor(rep(c("A", "B"), each = n))
  yb <- as.integer(y == "B")

  ## True Bayes-optimal log-probability at each sampled point. For two
  ## equal-variance Gaussians with equal priors, log(fB/fA) is exactly
  ## linear in x, so p_true(x) = plogis(a_true + b_true * x) -- same
  ## algebra as the two-Gaussian sigmoid discussed earlier in this project.
  b_true <- (mu2 - mu1) / sdv^2
  a_true <- -(mu2^2 - mu1^2) / (2 * sdv^2)
  eta_true  <- a_true + b_true * x
  log_p_true <- plogis(eta_true, log.p = TRUE)

  ## Is this replication's sample linearly separable in x, i.e. does some
  ## threshold perfectly split class A from class B?
  separable <- (max(x1) < min(x2)) || (max(x2) < min(x1))

  ## L1 / L2 penalized logistic regression (glmnet needs >= 2 predictor
  ## columns; standardize + pad with a zero column).
  ## type = "link" returns the linear predictor (log-odds) directly, so we
  ## never form the linear-scale probability.
  x_mean <- mean(x); x_sd <- sd(x)
  X <- cbind((x - x_mean) / x_sd, 0)
  cv_l2 <- cv.glmnet(X, y, family = "binomial", alpha = 0, standardize = FALSE)
  cv_l1 <- cv.glmnet(X, y, family = "binomial", alpha = 1, standardize = FALSE)
  eta_l2 <- as.numeric(predict(cv_l2, newx = X, s = "lambda.min", type = "link"))
  eta_l1 <- as.numeric(predict(cv_l1, newx = X, s = "lambda.min", type = "link"))

  ## Firth logistic regression.
  ## pl = FALSE skips profile-likelihood CIs, which we never use here and
  ## which otherwise emit harmless-but-noisy non-convergence warnings under
  ## strong separation.
  firth_fit <- logistf(yb ~ x, pl = FALSE)
  eta_firth <- coef(firth_fit)[1] + coef(firth_fit)[2] * x

  ## Platt scaling (standard and 2N-target variant). type = "link" again
  ## keeps everything on the log-odds scale.
  Np <- sum(yb == 1); Nn <- sum(yb == 0)
  t_std <- ifelse(yb == 1, (Np + 1) / (Np + 2), 1 / (Nn + 2))
  t_2N  <- ifelse(yb == 1, (2 * Np + 1) / (2 * Np + 2), 1 / (2 * Nn + 2))
  eta_platt  <- predict(suppressWarnings(glm(t_std ~ x, family = binomial)), type = "link")
  eta_platt2 <- predict(suppressWarnings(glm(t_2N ~ x, family = binomial)), type = "link")

  ## LDA. With one predictor, equal class priors, and a pooled within-class
  ## variance, LDA's posterior is exactly plogis(a_lda + b_lda * x) for the
  ## same reason p_true is a sigmoid above -- just with sample mean/variance
  ## in place of the true population parameters. Computing it this way
  ## (rather than via MASS::lda's predict(), which returns linear-scale
  ## posteriors) keeps it on the log scale too.
  m1 <- mean(x1); m2 <- mean(x2)
  s2_pooled <- ((n - 1) * var(x1) + (n - 1) * var(x2)) / (2 * n - 2)
  b_lda <- (m2 - m1) / s2_pooled
  a_lda <- -(m2^2 - m1^2) / (2 * s2_pooled)
  eta_lda <- a_lda + b_lda * x

  ## Bayesian logistic regression, posterior mode via arm::bayesglm, with
  ## Gelman, Jakulin, Pittau & Su's (2008) data-INDEPENDENT weakly
  ## informative default: independent Cauchy(0, 2.5) priors on the
  ## coefficients and Cauchy(0, 10) on the intercept, after bayesglm's
  ## internal rescaling of the predictor (scaled = TRUE, their
  ## recommendation). Unlike Firth's Jeffreys prior, this prior does not
  ## depend on the design matrix. prior.df = 1 is the Cauchy.
  cauchy_fit <- bayesglm(yb ~ x, family = binomial,
                         prior.scale = 2.5, prior.df = 1,
                         prior.scale.for.intercept = 10,
                         prior.df.for.intercept = 1)
  eta_cauchy <- coef(cauchy_fit)[1] + coef(cauchy_fit)[2] * x

  ## Same construction with a Student-t (df = 7) prior: lighter tails than
  ## the Cauchy (df = 1) but still well short of Normal, so it sits between
  ## Cauchy's robustness and L2/Gaussian shrinkage rather than collapsing
  ## toward L2 the way a higher df (close to the top of the commonly
  ## recommended 3-7 range) would.
  t_fit <- bayesglm(yb ~ x, family = binomial,
                    prior.scale = 2.5, prior.df = 7,
                    prior.scale.for.intercept = 10,
                    prior.df.for.intercept = 7)
  eta_t <- coef(t_fit)[1] + coef(t_fit)[2] * x

  etas <- list(L1 = eta_l1, L2 = eta_l2, Firth = eta_firth,
               Platt = eta_platt, Platt_2N = eta_platt2, LDA = eta_lda,
               Cauchy = eta_cauchy, StudentT = eta_t)

  do.call(rbind, lapply(names(etas), function(model_name) {
    log_p_hat <- plogis(etas[[model_name]], log.p = TRUE)
    data.frame(
      rep         = rep_id,
      point_id    = seq_along(x),
      class       = as.character(y),
      x           = x,
      model       = model_name,
      log_p_true  = log_p_true,
      log_p_hat   = log_p_hat,
      log10_ratio = (log_p_hat - log_p_true) / ln10,
      separable   = separable
    )
  }))
}

results <- do.call(rbind, lapply(seq_len(n_reps), fit_one_rep))
ratios_df <- results
M_reps <- length(unique(ratios_df$rep))
cat(sprintf("%d replications, %d of them linearly separable\n",
            M_reps, sum(tapply(ratios_df$separable, ratios_df$rep, all))))

### Figure 5: Multiplicative miscalibration in repeated simulations

Distribution of log10(predicted probability / true probability) for each regularized variant of logistic regression, restricted to class-A points and pooled across all Monte Carlo replications. The dashed red line marks exact agreement with the true probability (ratio = 1).

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 2.7)

model_order  <- c("Firth", "Platt", "Cauchy", "StudentT")
model_labels <- c(Firth = "Firth logistic", Platt = "Platt scaling",
                  Cauchy = "Cauchy(0, 2.5) prior", StudentT = "Student-t(7) prior")

par(mfrow = c(1, 4), mar = c(3.6, 2.2, 2.2, 0.6), mgp = c(2.1, 0.6, 0))
par(cex = 0.95, cex.axis = 0.95, cex.lab = 1, cex.main = 1.05)
## Common x-axis and bin width for all panels, so that the methods can be
## compared directly.
in_fig <- ratios_df$class == "A" & ratios_df$model %in% model_order
x_rng  <- range(c(0, ratios_df$log10_ratio[in_fig]))
br_all <- seq(floor(x_rng[1]), ceiling(x_rng[2]), by = 0.25)
for (m in model_order) {
  log_ratio <- ratios_df$log10_ratio[ratios_df$model == m & ratios_df$class == "A"]
  h <- hist(log_ratio, breaks = br_all, plot = FALSE)
  ymax <- max(h$density) * 1.15

  hist(log_ratio, breaks = br_all, freq = FALSE, col = "grey55", border = "grey55",
       xlab = "log10(pred. / true)", ylab = "", ylim = c(0, ymax),
       xlim = range(br_all), main = unname(model_labels[m]), yaxt = "n")
  axis(2, labels = FALSE)
  abline(v = 0, col = "red", lwd = 2, lty = 2)

  ## Small downward arrow at the mean, roughly mid-height, labeled above it.
  m_val <- mean(log_ratio)
  y_top <- ymax * 0.65
  y_bot <- ymax * 0.45
  arrows(x0 = m_val, y0 = y_top, x1 = m_val, y1 = y_bot,
         col = "blue", lwd = 2, length = 0.08)
  text(m_val, y_top, sprintf("mean=%.1f", m_val), col = "blue", pos = 3, cex = 1)
}

### Figure 6: Firth logistic regression as a function of class separation

For each Cohen's d from 1 to 8, 100 datasets of 50 samples per class from N(0, 1) and N(d, 1). (a) log10(predicted / true probability) of the less likely class at the mode of class A (x = 0), mean ± 1 SD. (b) Mean fitted and true odds ratio per 1 SD of x.

In [ ]:
options(repr.plot.width = 6.5, repr.plot.height = 6.5)

set.seed(1)

n_reps <- 100
n      <- 50
d_grid <- seq(1, 8, by = 0.25)

## Firth logistic regression via modified-score IRLS (same algorithm as the
## multi-dimensional version in the three_class_* scripts, here with a
## 1-D design). Self-contained: no logistf dependency.
firth_logistic_1d <- function(x, y, tol = 1e-8, max_iter = 100) {
  X <- cbind(1, x)
  beta <- c(0, 0)
  for (i in 1:max_iter) {
    eta  <- as.vector(X %*% beta)
    p    <- plogis(eta)
    W    <- p * (1 - p)
    XtWX <- t(X) %*% (X * W)
    XtWX_inv <- solve(XtWX)
    h <- W * rowSums((X %*% XtWX_inv) * X)
    U_star <- t(X) %*% (y - p + h * (0.5 - p))
    step <- as.vector(XtWX_inv %*% U_star)
    beta <- beta + step
    if (max(abs(step)) < tol) break
  }
  beta
}

ln10 <- log(10)

res <- do.call(rbind, lapply(d_grid, function(d) {
  per_rep <- vapply(seq_len(n_reps), function(rep_id) {
    x <- c(rnorm(n, 0, 1), rnorm(n, d, 1))
    y <- rep(c(0, 1), each = n)
    beta <- firth_logistic_1d(x, y)
    ## Predicted and true log P(B | x = 0); x = 0 is the mode of class A,
    ## where B is the less likely class. eta_true(0) = -d^2/2.
    log_p_hat  <- plogis(beta[1], log.p = TRUE)
    log_p_true <- plogis(-d^2 / 2, log.p = TRUE)
    ## beta[2] is the fitted log odds ratio per 1 SD of x (the true value
    ## under the generative model is d, i.e. true OR = exp(d)).
    c(ratio = (log_p_hat - log_p_true) / ln10, log_or = beta[2])
  }, c(ratio = 0, log_or = 0))
  data.frame(d = d,
             mean_log10_ratio = mean(per_rep["ratio", ]),
             sd_log10_ratio   = sd(per_rep["ratio", ]),
             mean_log10_or    = mean(per_rep["log_or", ]) / ln10,
             true_log10_or    = d / ln10)
}))


## Two stacked panels sharing the d axis (instead of one panel with a second
## y-axis): (a) the miscalibration at the mode of class A, (b) the fitted and
## true log odds ratio per 1 SD of x.
col_firth <- "#7570b3"; col_or <- "#1b9e77"
layout(matrix(1:2, ncol = 1), heights = c(1, 0.8))
xl <- range(res$d)

## (a) Miscalibration.
par(mar = c(1.2, 4.8, 1.6, 1), mgp = c(2.8, 0.7, 0))
ylim_a <- range(res$mean_log10_ratio - res$sd_log10_ratio,
                res$mean_log10_ratio + res$sd_log10_ratio, 0)
plot(res$d, res$mean_log10_ratio, type = "n", xlim = xl, ylim = ylim_a,
     xaxt = "n", las = 1, xlab = "",
     ylab = "log10(predicted / true prob.)")
axis(1, labels = FALSE)
polygon(c(res$d, rev(res$d)),
        c(res$mean_log10_ratio - res$sd_log10_ratio,
          rev(res$mean_log10_ratio + res$sd_log10_ratio)),
        col = adjustcolor(col_firth, alpha.f = 0.25), border = NA)
lines(res$d, res$mean_log10_ratio, lwd = 2, col = col_firth)
abline(h = 0, col = "red", lwd = 2, lty = 2)
mtext("(a) Miscalibration at the mode of class A", side = 3, line = 0.3, adj = 0, font = 2)
legend("topleft", bty = "n", cex = 0.9,
       legend = c("Firth, mean of 100 datasets", expression(""%+-%"1 SD"),
                  "perfect calibration"),
       col = c(col_firth, adjustcolor(col_firth, alpha.f = 0.25), "red"),
       lty = c(1, NA, 2), lwd = c(2, NA, 2), pch = c(NA, 15, NA), pt.cex = c(NA, 2, NA))

## (b) Odds ratio per 1 SD of x, on a log scale.
par(mar = c(4.2, 4.8, 1.6, 1))
or_ticks <- c(1, 3, 10, 30, 100, 300, 1000, 3000)
ylim_b <- range(0, res$true_log10_or, res$mean_log10_or)
plot(res$d, res$true_log10_or, type = "n", xlim = xl, ylim = ylim_b,
     yaxt = "n", xlab = expression("Cohen's " * italic(d)),
     ylab = expression("odds ratio per 1 SD of " * italic(x)))
axis(2, at = log10(or_ticks), labels = or_ticks, las = 1)
lines(res$d, res$true_log10_or, lwd = 2, lty = 3, col = "black")
lines(res$d, res$mean_log10_or, lwd = 2, col = col_or)
mtext("(b) Fitted and true odds ratio", side = 3, line = 0.3, adj = 0, font = 2)
legend("topleft", bty = "n", cex = 0.9,
       legend = c(expression("true OR = exp(" * italic(d) * ")"), "Firth fitted OR (mean)"),
       col = c("black", col_or), lty = c(3, 1), lwd = 2)

## Part 3: speaker embeddings

The data are 192-dimensional ECAPA-TDNN embeddings of 400 ms clips of three speakers (A, M, S) reading two stories (EN, EB). The first 70% of each recording is used for training and the last 30% for testing. Columns: `speaker`, `story`, `set` (train/test), `start_ms` (start of the clip within the recording), `dim_001` ... `dim_192`.

The next cell projects the embeddings onto two discriminant directions and fits the pairwise LDA and Firth classifiers.

In [ ]:
## ---------------------------------------------------------------------------
## Partial one-vs-one coupling on speaker embeddings: both English fables,
## 400 ms non-overlapping clips, each recording split 70/30 in time (first 70 %
## of its speech -> training, last 30 % -> testing) before cutting.
## Embeddings: ECAPA-TDNN, 192 dimensions, in data/ecapa_embeddings.csv
## ---------------------------------------------------------------------------
emb <- read.csv("data/ecapa_embeddings.csv")
emb$speaker <- factor(emb$speaker)
X192 <- as.matrix(emb[, grep("^dim_", names(emb))])
spk  <- levels(emb$speaker)                                    # "A" "M" "S"
is_train <- emb$set == "train"
Xtr192 <- X192[is_train, ];  ytr <- emb$speaker[is_train]
Xte192 <- X192[!is_train, ]; yte <- emb$speaker[!is_train]
p192 <- ncol(X192); K3 <- 3
n_tr <- nrow(Xtr192); n_te <- nrow(Xte192)

## Regularized LDA projection 192 -> K-1 = 2, fitted on the training data only.
## S_w is shrunk toward a scaled identity before inversion because n only
## moderately exceeds p; whitening then makes the pooled within-class covariance
## spherical.
alpha3 <- 0.5
nk3    <- as.integer(table(ytr))
mu192  <- t(sapply(spk, function(k) colMeans(Xtr192[ytr == k, ])))
Sw192  <- Reduce(`+`, lapply(1:K3, function(i)
            (nk3[i] - 1) * cov(Xtr192[ytr == spk[i], ]))) / (nrow(Xtr192) - K3)
Sw192  <- (1 - alpha3) * Sw192 + alpha3 * mean(diag(Sw192)) * diag(p192)
ew192  <- eigen(Sw192, symmetric = TRUE)
Wh192  <- ew192$vectors %*% diag(1 / sqrt(pmax(ew192$values, 1e-12))) %*% t(ew192$vectors)
V2     <- eigen(cov(mu192 %*% Wh192) * (K3 - 1), symmetric = TRUE)$vectors[, 1:2]
P2     <- Wh192 %*% V2

Ztr <- Xtr192 %*% P2; Zte <- Xte192 %*% P2
means3 <- t(sapply(spk, function(k) colMeans(Ztr[ytr == k, ])))
dimnames(means3) <- list(spk, c("x", "y"))
dat3  <- data.frame(x = Ztr[, 1], y = Ztr[, 2], class = ytr)   # training
test3 <- data.frame(x = Zte[, 1], y = Zte[, 2], class = yte)   # test, plotted

## ---- properties of the reduced data ---------------------------------------
royston_H <- function(X) {
  X <- as.matrix(X); n <- nrow(X); p <- ncol(X)
  kurt <- function(v) { m <- mean(v); mean((v - m)^4) / mean((v - m)^2)^2 }
  sf_W <- function(v) cor(sort(v), qnorm(ppoints(length(v), a = 3/8)))^2
  lx <- log(n)
  mu <- -1.5861 - 0.31082 * lx - 0.083751 * lx^2 + 0.0038915 * lx^3
  sg <- exp(-0.4803 - 0.082676 * lx + 0.0030302 * lx^2)
  z <- vapply(seq_len(p), function(i) {
    v <- X[, i]
    W <- if (kurt(v) > 3) sf_W(v) else as.numeric(shapiro.test(v)$statistic)
    (log(1 - W) - mu) / sg
  }, numeric(1))
  u <- 0.715; v <- 0.21364 + 0.015124 * lx^2 - 0.0018034 * lx^3; l <- 5
  C  <- cor(X); NC <- (C^l) * (1 - (u * (1 - C)^u) / v)
  mC <- (sum(NC) - p) / (p^2 - p); edf <- p / (1 + (p - 1) * mC)
  H  <- edf * sum(qnorm(pnorm(-z) / 2)^2) / p
  pchisq(H, df = edf, lower.tail = FALSE)
}
box_M <- function(Z, g) {
  lv <- levels(droplevels(g)); gq <- length(lv); m <- nrow(Z); q <- ncol(Z)
  ni <- as.integer(table(g)[lv])
  Si <- lapply(lv, function(k) cov(Z[g == k, , drop = FALSE]))
  Sp <- Reduce(`+`, Map(function(S, a) (a - 1) * S, Si, ni)) / (m - gq)
  M  <- (m - gq) * log(det(Sp)) - sum((ni - 1) * sapply(Si, function(S) log(det(S))))
  c1 <- (sum(1 / (ni - 1)) - 1 / (m - gq)) * (2 * q^2 + 3 * q - 1) / (6 * (q + 1) * (gq - 1))
  pchisq(M * (1 - c1), (gq - 1) * q * (q + 1) / 2, lower.tail = FALSE)
}
mahalanobis_dist <- function(Z, g, a, b) {
  i <- g == a; j <- g == b
  S  <- ((sum(i) - 1) * cov(Z[i, ]) + (sum(j) - 1) * cov(Z[j, ])) / (sum(i) + sum(j) - 2)
  dm <- colMeans(Z[i, ]) - colMeans(Z[j, ])
  sqrt(as.numeric(t(dm) %*% solve(S) %*% dm))
}
separable_pair <- function(Z, g, a, b) {
  idx <- g %in% c(a, b); yy <- as.integer(g[idx] == b)
  fit <- suppressWarnings(glm(yy ~ Z[idx, ], family = binomial))
  all((fitted(fit) > 0.5) == (yy == 1))
}
roy_tr <- sapply(spk, function(k) royston_H(Ztr[ytr == k, ]))
roy_te <- sapply(spk, function(k) royston_H(Zte[yte == k, ]))
boxM_tr <- box_M(Ztr, ytr); boxM_te <- box_M(Zte, yte)
pair_names <- sapply(combn(spk, 2, simplify = FALSE), paste, collapse = "-")
d_tr <- setNames(sapply(combn(spk, 2, simplify = FALSE), function(pr) mahalanobis_dist(Ztr, ytr, pr[1], pr[2])), pair_names)
d_te <- setNames(sapply(combn(spk, 2, simplify = FALSE), function(pr) mahalanobis_dist(Zte, yte, pr[1], pr[2])), pair_names)
sep_tr <- setNames(sapply(combn(spk, 2, simplify = FALSE), function(pr) separable_pair(Ztr, ytr, pr[1], pr[2])), pair_names)
sep_te <- setNames(sapply(combn(spk, 2, simplify = FALSE), function(pr) separable_pair(Zte, yte, pr[1], pr[2])), pair_names)

pairs3 <- combn(3, 2, simplify = FALSE)   # (1,2) (1,3) (2,3) indexing spk

## LDA pairwise fits: closed-form pooled-covariance discriminant, evaluated
## as a raw linear score (not via predict.lda) so it stays informative at
## full precision far from the training data.
lda_pair_coef <- list()
for (pr in pairs3) {
  lo <- pr[1]; hi <- pr[2]
  Xlo <- as.matrix(dat3[as.integer(dat3$class) == lo, c("x", "y")])
  Xhi <- as.matrix(dat3[as.integer(dat3$class) == hi, c("x", "y")])
  n_lo <- nrow(Xlo); n_hi <- nrow(Xhi)
  mlo <- colMeans(Xlo); mhi <- colMeans(Xhi)
  Sigma <- ((n_lo - 1) * cov(Xlo) + (n_hi - 1) * cov(Xhi)) / (n_lo + n_hi - 2)
  Sinv  <- solve(Sigma)
  w <- as.numeric(Sinv %*% (mhi - mlo))
  b <- -0.5 * as.numeric(mhi %*% Sinv %*% mhi - mlo %*% Sinv %*% mlo) + log(n_hi / n_lo)
  lda_pair_coef[[paste0(lo, "_", hi)]] <- list(w = w, b = b)
}
lda_eta_pair_raw <- function(lo, hi, Xnew) {
  cf <- lda_pair_coef[[paste0(lo, "_", hi)]]
  as.numeric(Xnew %*% cf$w + cf$b)
}

## Firth pairwise fits (bias-reduced logistic regression via modified-score
## IRLS), evaluated as a raw linear predictor for the same reason.
firth_logistic3 <- function(X, y, tol = 1e-8, max_iter = 200) {
  X <- cbind(Intercept = 1, X)
  beta <- rep(0, ncol(X))
  for (i in 1:max_iter) {
    eta  <- as.vector(X %*% beta)
    p_i  <- plogis(eta)
    W    <- p_i * (1 - p_i)
    XtWX <- t(X) %*% (X * W)
    XtWX_inv <- solve(XtWX)
    h <- W * rowSums((X %*% XtWX_inv) * X)
    U_star <- t(X) %*% (y - p_i + h * (0.5 - p_i))
    step <- as.vector(XtWX_inv %*% U_star)
    beta <- beta + step
    if (max(abs(step)) < tol) break
  }
  list(beta = beta)
}
firth_pair_fits <- list()
for (pr in pairs3) {
  lo <- pr[1]; hi <- pr[2]
  sub <- dat3[as.integer(dat3$class) %in% c(lo, hi), ]
  Xp <- as.matrix(sub[, c("x", "y")])
  yp <- as.integer(as.integer(sub$class) == hi)
  firth_pair_fits[[paste0(lo, "_", hi)]] <- firth_logistic3(Xp, yp)
}
firth_eta_pair_raw <- function(lo, hi, Xnew) {
  fit <- firth_pair_fits[[paste0(lo, "_", hi)]]
  as.numeric(cbind(1, Xnew) %*% fit$beta)
}

## Signed log-odds of class a over class b (a != b, either order), for a
## given base classifier's raw pairwise score function.
make_log_odds <- function(eta_pair_raw) {
  function(a, b, Xnew) {
    lo <- min(a, b); hi <- max(a, b)
    eta <- eta_pair_raw(lo, hi, Xnew)
    if (a == hi) eta else -eta
  }
}
lda_log_odds   <- make_log_odds(lda_eta_pair_raw)
firth_log_odds <- make_log_odds(firth_eta_pair_raw)

## Partial model, closed form (see prose): p_i : p_j : p_k = exp(eta_ik) : exp(eta_jk) : 1,
## evaluated with the softmax stabilization on the raw log-odds.
couple_predict_partial <- function(Xnew, i, j, k, log_odds_fn) {
  eta_i <- log_odds_fn(i, k, Xnew)
  eta_j <- log_odds_fn(j, k, Xnew)
  L <- matrix(0, nrow(Xnew), 3)
  L[, i] <- eta_i; L[, j] <- eta_j; L[, k] <- 0
  w <- exp(L - apply(L, 1, max))
  w / rowSums(w)
}

## Full model: majority voting of the three pairwise classifiers; each casts
## one vote for the class its log-odds favor. Returns a vote-count matrix,
## so that acc() below (argmax per row) applies unchanged.
vote_predict_full <- function(Xnew, log_odds_fn) {
  v <- matrix(0, nrow(Xnew), 3)
  for (pr in pairs3) {
    a <- pr[1]; b <- pr[2]; w <- log_odds_fn(a, b, Xnew) > 0
    v[w, a] <- v[w, a] + 1; v[!w, b] <- v[!w, b] + 1
  }
  v
}

drops <- list(drop12 = c(i = 1, j = 2, k = 3),
              drop13 = c(i = 1, j = 3, k = 2),
              drop23 = c(i = 2, j = 3, k = 1))
drop_titles <- c(drop12 = sprintf("drop pair (%s,%s)", spk[1], spk[2]),
                 drop13 = sprintf("drop pair (%s,%s)", spk[1], spk[3]),
                 drop23 = sprintf("drop pair (%s,%s)", spk[2], spk[3]))

## ---- held-out accuracies ----------------------------------------------------
Xte2  <- as.matrix(test3[, c("x", "y")])
truth <- as.integer(test3$class)
acc <- function(P) mean(max.col(P) == truth)
acc_full  <- c(LDA = acc(vote_predict_full(Xte2, lda_log_odds)),
               Firth = acc(vote_predict_full(Xte2, firth_log_odds)))
acc_drop  <- sapply(drops, function(d) c(
  LDA   = acc(couple_predict_partial(Xte2, d[["i"]], d[["j"]], d[["k"]], lda_log_odds)),
  Firth = acc(couple_predict_partial(Xte2, d[["i"]], d[["j"]], d[["k"]], firth_log_odds))))
lda_max_change <- max(abs(acc_drop["LDA", ] - acc_full["LDA"]))
pred_dropAS <- max.col(couple_predict_partial(Xte2, 1, 3, 2, firth_log_odds))
n_S_to_A <- sum(truth == 3 & pred_dropAS == 1)

## ---- linkage-class miscalibration (fig-miscal-box) -----------------------
## log10 p(k | x) from the pairwise classifier of pair (c, k), on the log scale.
log10_pk <- function(c, k, X, eta_fn) {
  lo <- min(c, k); hi <- max(c, k)
  sgn <- if (k == hi) 1 else -1                      # log-odds of k over c
  plogis(sgn * eta_fn(lo, hi, X), log.p = TRUE) / log(10)
}
## For each dropped pair (i, j) with linkage class k, both remaining classifiers
## are evaluated on the same test points, those of one class of the pair (eval_cls;
## see the figure caption). The first box is the classifier of the other class,
## the second that of the examined class.
eval_cls <- c(drop12 = match("M", spk), drop13 = match("S", spk), drop23 = match("M", spk))
link <- list(); link_diff <- numeric(0); link_lab <- character(0); link_acc <- numeric(0)
for (dn in names(drops)) {
  d <- drops[[dn]]; i <- d[["i"]]; j <- d[["j"]]; k <- d[["k"]]
  pc <- eval_cls[[dn]]; other <- if (pc == i) j else i
  X <- Xte2[truth == pc, , drop = FALSE]
  for (c in c(other, pc)) {
    link[[length(link) + 1]] <- list(
      v = log10_pk(c, k, X, firth_eta_pair_raw) - log10_pk(c, k, X, lda_eta_pair_raw),
      cls = c, lab = sprintf("%s-%s\nclassifier", spk[c], spk[k]))
  }
  n <- length(link)
  link_diff <- c(link_diff, median(link[[n]]$v) - median(link[[n - 1]]$v))
  link_lab  <- c(link_lab, sprintf("drop %s-%s (linkage %s)\non %s points", spk[i], spk[j], spk[k], spk[pc]))
  sel <- truth %in% c(i, j)      # accuracy on the test clips of the two dropped classes
  link_acc  <- c(link_acc, mean(max.col(couple_predict_partial(Xte2[sel, ], i, j, k, firth_log_odds)) == truth[sel]))
}
names(link_diff) <- names(link_acc) <- names(drops)
## How well do the medians in fig-miscal-box describe the distortion of the
## odds p_i/p_j? link_small: smallest share, over the two classifiers, of the
## evaluated points with p(k) < 1/2 (Firth); link_exact: median signed (toward
## misclassification)
## Firth-minus-LDA error of log10(p_i/p_j), computed exactly from eq-odds-linkage.
link_small <- link_exact <- link_R <- link_share <- link_lda_own <- link_lda_for <- setNames(numeric(3), names(drops)); link_exact_v <- link_R_v <- list()
for (dn in names(drops)) {
  d <- drops[[dn]]; i <- d[["i"]]; j <- d[["j"]]; k <- d[["k"]]
  X <- Xte2[truth == eval_cls[[dn]], , drop = FALSE]
  link_small[dn] <- min(sapply(c(i, j), function(c) mean(firth_log_odds(k, c, X) < 0)))
  pc <- eval_cls[[dn]]; other <- if (pc == i) j else i
  ## Firth-minus-LDA error of log10(p_other / p_pc) at points of class pc, from
  ## eq-odds-linkage: p_other/p_pc = (p_other/p_k) / (p_pc/p_k).
  link_exact_v[[dn]] <- ((firth_log_odds(other, k, X) - firth_log_odds(pc, k, X)) -
                         (lda_log_odds(other, k, X) - lda_log_odds(pc, k, X))) / log(10)
  link_exact[dn] <- median(link_exact_v[[dn]])
  ## eq-delta-decomposition: remainder R = Delta - (second box - first box).
  g <- match(dn, names(drops))
  link_R_v[[dn]] <- link_exact_v[[dn]] - (link[[2 * g]]$v - link[[2 * g - 1]]$v)
  ## Share of the mean of Delta carried by D_ck (the classifier of the evaluated class),
  ## and the median LDA log10 p(k) under the classifiers c-k (own) and c'-k (foreign).
  link_share[dn] <- mean(link[[2 * g]]$v) / mean(link_exact_v[[dn]])
  link_lda_own[dn] <- median(log10_pk(pc, k, X, lda_eta_pair_raw))
  link_lda_for[dn] <- median(log10_pk(other, k, X, lda_eta_pair_raw))
  link_R[dn] <- mean(abs(link_R_v[[dn]]) < 0.1)   # share of points with |R| < 0.1
}
## Robustness of the arbitrary choice for the (M,S)-dropped case: the median
## distortion when the S test points are evaluated instead of the M points.
alt23 <- local({
  d <- drops$drop23; k <- d[["k"]]; pc <- match("S", spk); other <- match("M", spk)
  X <- Xte2[truth == pc, , drop = FALSE]
  median(((firth_log_odds(other, k, X) - firth_log_odds(pc, k, X)) -
          (lda_log_odds(other, k, X) - lda_log_odds(pc, k, X))) / log(10))
})

f2 <- function(x) sprintf("%.2f", x)
f3 <- function(x) sprintf("%.3f", x)
pct <- function(x) sprintf("%.1f", 100 * x)

### Figure 7: Decision regions of the partial models

Argmax decision regions of the partial models, dropping one pair at a time (rows), with LDA (left) and Firth logistic regression (right) as base classifiers. Each panel gives the held-out accuracy and the speaker whose test clips are most often misclassified. Solid half-lines are the boundaries of the two pairwise classifiers used by the partial model; the dotted half-line is the boundary of the dropped pair's own classifier, which the partial model does not use. Crosses mark the training class means; LD1 and LD2 are the canonical discriminant directions.

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 10)


## Okabe-Ito colours: distinguishable for colour-blind readers and in greyscale.
class_cols3 <- c("#E69F00", "#009E73", "#0072B2")   # A orange, M green, S blue

## Plot window: the data and the three means with a small margin.
allxy <- rbind(as.matrix(dat3[, c("x", "y")]), as.matrix(test3[, c("x", "y")]), means3)
pad3  <- 0.6
xlim3 <- range(allxy[, 1]) + c(-pad3, pad3)
ylim3 <- range(allxy[, 2]) + c(-pad3, pad3)

## Region grid over a given rectangle (the full plotting area of a panel, so
## the shading reaches every edge despite the equal-aspect constraint).
region_grid <- function(usr, n = 220) {
  gx <- seq(usr[1], usr[2], length.out = n)
  gy <- seq(usr[3], usr[4], length.out = n)
  list(gx = gx, gy = gy, X = as.matrix(expand.grid(x = gx, y = gy)))
}

## One panel: shaded argmax regions of the partial model, pairwise boundaries,
## test clips (misclassified ones circled), training means.
plot_argmax_panel <- function(i, j, k, log_odds_fn, model_label, acc_value) {
  plot.new()
  plot.window(xlim3, ylim3, xaxs = "i", yaxs = "i")
  g <- region_grid(par("usr"))
  pred_grid  <- max.col(couple_predict_partial(g$X, i, j, k, log_odds_fn))
  image(g$gx, g$gy, matrix(pred_grid, length(g$gx)), add = TRUE, useRaster = TRUE,
        col = adjustcolor(class_cols3, alpha.f = 0.25), breaks = c(0.5, 1.5, 2.5, 3.5))
  ## Pairwise boundaries of this model type, drawn as half-lines: the part of the
  ## line a|b on which both a and b beat the third class c (by the pairwise
  ## classifiers a|c and b|c). For LDA the three half-lines meet in one point.
  usr <- par("usr")
  for (pr in pairs3) {
    a <- pr[1]; b <- pr[2]; cc <- setdiff(1:3, c(a, b)); key <- paste0(a, "_", b)
    if (model_label == "LDA") { w <- lda_pair_coef[[key]]$w; b0 <- lda_pair_coef[[key]]$b
    } else { be <- firth_pair_fits[[key]]$beta; b0 <- be[1]; w <- be[2:3] }
    p0 <- -b0 * w / sum(w^2); dvec <- c(-w[2], w[1]) / sqrt(sum(w^2))
    ## t-interval where a beats c and b beats c (each condition is linear in t)
    lo_t <- -Inf; hi_t <- Inf
    for (u in c(a, b)) {
      f0 <- log_odds_fn(u, cc, matrix(p0, 1))
      f1 <- log_odds_fn(u, cc, matrix(p0 + dvec, 1)) - f0
      if (abs(f1) < 1e-12) { if (f0 <= 0) { lo_t <- Inf } ; next }
      if (f1 > 0) lo_t <- max(lo_t, -f0 / f1) else hi_t <- min(hi_t, -f0 / f1)
    }
    if (lo_t >= hi_t) next
    big <- 1e3; lo_t <- max(lo_t, -big); hi_t <- min(hi_t, big)
    x1 <- p0 + lo_t * dvec; x2 <- p0 + hi_t * dvec
    dropped <- setequal(c(a, b), c(i, j))
    segments(x1[1], x1[2], x2[1], x2[2], lwd = 1.8, lty = if (dropped) 3 else 1)
    ## label near the far end of the half-line, inside the frame
    tt <- seq(lo_t, hi_t, length.out = 2000); pts <- cbind(p0[1] + tt * dvec[1], p0[2] + tt * dvec[2])
    ins <- which(pts[, 1] > usr[1] & pts[, 1] < usr[2] & pts[, 2] > usr[3] & pts[, 2] < usr[4])
    if (length(ins) == 0) next
    start <- if (lo_t > -big) lo_t else hi_t          # the half-line's starting point
    dd <- abs(tt[ins] - start)
    ok <- pts[ins, 2] < usr[4] - 0.2 * diff(usr[3:4])   # keep clear of the accuracy box
    if (!any(ok)) next
    q <- pts[ins[ok][which.min(abs(dd[ok] - 0.85 * max(dd[ok])))], ]
    lab <- sprintf("%s|%s", spk[a], spk[b])
    lw <- strwidth(lab, cex = 0.8, font = 2); lh <- strheight(lab, cex = 0.8, font = 2)
    rect(q[1] - 0.6 * lw, q[2] - 0.75 * lh, q[1] + 0.6 * lw, q[2] + 0.75 * lh, col = "white", border = NA)
    text(q[1], q[2], lab, cex = 0.8, font = 2)
  }
  pred_test <- max.col(couple_predict_partial(Xte2, i, j, k, log_odds_fn))
  wrong <- pred_test != truth
  points(test3$x, test3$y, col = class_cols3[truth], pch = 16, cex = 0.75)
  points(means3[, 1], means3[, 2], pch = 4, lwd = 3, cex = 1.6)
  box()
  title(xlab = "LD1", ylab = "LD2", line = 0.5)
  title(main = sprintf("%s, drop pair (%s, %s)", model_label, spk[i], spk[j]),
        line = 0.5, cex.main = 1.15)
  ## Accuracy, inside the frame at the top, on a white background.
  usr <- par("usr"); lab <- sprintf("Accuracy %d%%", round(100 * acc_value))
  w <- strwidth(lab, cex = 0.95); h <- strheight(lab, cex = 0.95)
  xm <- mean(usr[1:2]); ym <- usr[4] - 0.06 * diff(usr[3:4])
  rect(xm - 0.6 * w, ym - 0.9 * h, xm + 0.6 * w, ym + 0.9 * h,
       col = "white", border = "grey40", lwd = 0.8)
  text(xm, ym, lab, cex = 0.95)
  ## Below the frame: the speaker whose test clips are most often misclassified.
  per_class <- sapply(1:3, function(cl) mean(wrong[truth == cl]))
  worst <- which.max(per_class)
  if (per_class[worst] > 0)
    mtext(sprintf("%d%% samples of %s misclassified", round(100 * per_class[worst]), spk[worst]),
          side = 1, line = 1.7, cex = 0.85)
}

## Layout: a legend row spanning both columns on top, then the 3x2 grid.
par(oma = c(0.5, 0, 0, 0))
layout(matrix(c(1, 1,
                2, 3,
                4, 5,
                6, 7), nrow = 4, byrow = TRUE),
       heights = c(0.55, 3, 3, 3))

par(mar = c(0, 0, 0, 0))
plot.new()
legend("center", bty = "n", cex = 1.45, ncol = 3,
       legend = c(paste("speaker", spk), "training mean", "pairwise boundary", "dropped pair's boundary"),
       col = c(class_cols3, "black", "black", "black"),
       pch = c(16, 16, 16, 4, NA, NA), pt.cex = c(1.2, 1.2, 1.2, 1.3, NA, NA),
       pt.lwd = c(1, 1, 1, 3, NA, NA), lty = c(NA, NA, NA, NA, 1, 3), lwd = 1.8,
       seg.len = 2.2, x.intersp = 0.8, text.width = 0.24)

par(mar = c(3.0, 2.2, 1.9, 0.8), mgp = c(2.3, 0.7, 0), cex = 0.95, cex.lab = 1.1)
for (dn in names(drops)) {
  d <- drops[[dn]]
  plot_argmax_panel(d[["i"]], d[["j"]], d[["k"]], lda_log_odds,   "LDA",   acc_drop["LDA", dn])
  plot_argmax_panel(d[["i"]], d[["j"]], d[["k"]], firth_log_odds, "Firth", acc_drop["Firth", dn])
}

### Figure 8: Distortion of the odds

Distortion $\Delta(\mathbf{x})$ (@eq-odds-distortion) of the odds between the two classes of the dropped pair $(i,j)$, evaluated on the test points. The label under each box gives the odds $p_{c^\prime}/p_c$. The dashed red line marks exact agreement with LDA.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.3)


layout(1)   # reset the multi-panel layout left over from fig-partial-coupling
ex <- link_exact_v[names(drops)]
ex_lab <- sapply(names(drops), function(dn) {
  d <- drops[[dn]]; pc <- eval_cls[[dn]]; other <- if (pc == d[["i"]]) d[["j"]] else d[["i"]]
  sprintf("odds %s/%s", spk[other], spk[pc])
})
top <- max(unlist(ex)); bot <- min(0, unlist(ex))
par(mar = c(5.8, 4.8, 1.2, 1))
boxplot(ex, at = 1:3, xlim = c(0.4, 3.6), ylim = c(bot, top * 1.15), boxwex = 0.5,
        names = FALSE, las = 1, col = adjustcolor("#9467BD", alpha.f = 0.75),
        ylab = expression(Delta(bold(x))))
abline(h = 0, col = "red", lwd = 2, lty = 2)
text(1:3, sapply(ex, max), sprintf("median = %.2f", sapply(ex, median)), pos = 3, cex = 0.8)
mtext(ex_lab, side = 1, line = 1.0, at = 1:3, cex = 0.8)
mtext(link_lab, side = 1, line = 3.2, at = 1:3, cex = 0.8, font = 2)

### Figure 9: Multiplicative miscalibration of the linkage class

Multiplicative miscalibration of the linkage class $k$ when the pair $(c, c^\prime)$ is dropped: $D_{c^\prime k}(\mathbf{x})$ (first box of each panel) and $D_{ck}(\mathbf{x})$ (second box), defined in @eq-linkage-miscal. Both remaining classifiers are evaluated on the same test points as in @fig-odds-distortion. The dashed red line marks  agreement with LDA.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4.2)


pos <- c(1, 2, 4, 5, 7, 8)
top <- max(sapply(link, function(l) max(l$v)))
bot <- min(0, sapply(link, function(l) min(l$v)))
par(mar = c(7.2, 4.8, 1.5, 1))
boxplot(lapply(link, `[[`, "v"), at = pos, xlim = c(0.4, 8.6), ylim = c(bot, top * 1.04),
        names = FALSE, las = 1,
        col = adjustcolor(class_cols3[sapply(link, `[[`, "cls")], alpha.f = 0.55),
        ylab = expression(D[c*minute*k](bold(x))*","~~D[ck](bold(x))))
mtext(sapply(link, `[[`, "lab"), side = 1, line = 1.8, at = pos, cex = 0.75)
abline(h = 0, col = "red", lwd = 2, lty = 2)
abline(v = c(3, 6), col = "grey80")
for (g in seq_along(link_diff)) {
  x1 <- pos[2 * g - 1]; x2 <- pos[2 * g]
  mtext(link_lab[g], side = 1, line = 4.6, at = mean(c(x1, x2)), cex = 0.8, font = 2)
}

### Figure 10: Remainder of the decomposition

Distribution of the remainder $R(\mathbf{x})$ of @eq-delta-decomposition on the same test points as in @fig-odds-distortion and @fig-miscal-box, one dot per point. Where $R$ is close to zero (dashed red line), $\Delta$ equals the difference between the two boxes of the corresponding panel of @fig-miscal-box.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.6)


Rv <- link_R_v[names(drops)]
top <- max(unlist(Rv)); bot <- min(0, unlist(Rv))
par(mar = c(4.2, 4.8, 1.2, 1))
boxplot(Rv, at = 1:3, xlim = c(0.4, 3.6), ylim = c(bot, top * 1.15), boxwex = 0.5,
        names = FALSE, las = 1, col = "grey80", outline = FALSE,
        ylab = expression(R(bold(x))))
stripchart(Rv, vertical = TRUE, method = "jitter", jitter = 0.12, add = TRUE,
           pch = 16, cex = 0.5, col = adjustcolor("black", 0.45))
abline(h = 0, col = "red", lwd = 2, lty = 2)
mtext(link_lab, side = 1, line = 1.6, at = 1:3, cex = 0.8, font = 2)

### Temperature scaling

Blocked cross-validation of the two temperatures (20 runs each for 10 and 3 blocks).

In [ ]:
## Temperature scaling of the two remaining Firth classifiers when (A,S) is
## dropped (i = A, j = S, linkage k = M). Temperatures are chosen by blocked
## cross-validation on the training clips only; the test clips are used once.
tA <- match("A", spk); tM <- match("M", spk); tS <- match("S", spk)
te_sc <- cbind(AM = firth_log_odds(tA, tM, Xte2), SM = firth_log_odds(tS, tM, Xte2))
coupled_T <- function(sc, uAM, uSM) {             # log-odds / T, columns in spk order
  L <- matrix(0, nrow(sc), 3)
  L[, tA] <- sc[, "AM"] / exp(uAM); L[, tS] <- sc[, "SM"] / exp(uSM)
  L
}
test_acc_T <- function(uAM, uSM) mean(max.col(coupled_T(te_sc, uAM, uSM)) == truth)

## Test accuracy over a grid of log temperatures (for the figure only).
u_grid <- seq(-1.5, 1.5, length.out = 121)
acc_T_grid <- outer(u_grid, u_grid, Vectorize(test_acc_T))

## Out-of-fold log-odds: refit the projection and both Firth classifiers
## without one block of training clips, and score the held-out block.
proj_fit <- function(X, y) {
  nk <- as.integer(table(y)); mu <- t(sapply(spk, function(k) colMeans(X[y == k, ])))
  Sw <- Reduce(`+`, lapply(1:3, function(i) (nk[i] - 1) * cov(X[y == spk[i], ]))) / (nrow(X) - 3)
  Sw <- (1 - alpha3) * Sw + alpha3 * mean(diag(Sw)) * diag(ncol(X))
  e  <- eigen(Sw, symmetric = TRUE)
  Wh <- e$vectors %*% diag(1 / sqrt(pmax(e$values, 1e-12))) %*% t(e$vectors)
  Wh %*% eigen(cov(mu %*% Wh) * 2, symmetric = TRUE)$vectors[, 1:2]
}
oof_scores <- function(Xa, ya, Xb) {
  P <- proj_fit(Xa, ya); Za <- Xa %*% P; Zb <- Xb %*% P; g <- as.integer(ya)
  lo <- function(c) { s <- g %in% c(c, tM)
    b <- firth_logistic3(Za[s, ], as.integer(g[s] == c))$beta
    as.numeric(cbind(1, Zb) %*% b) }          # log-odds of c over M
  cbind(AM = lo(tA), SM = lo(tS))
}
## Contiguous blocks in time within each recording; the sequence is rotated by
## a random offset so that the block boundaries vary between runs.
tr_meta <- emb[is_train, c("speaker", "story", "start_ms")]
rec <- paste(tr_meta$speaker, tr_meta$story)
block_folds <- function(K, shift) {
  fd <- integer(nrow(tr_meta))
  for (r in unique(rec)) {
    ix <- which(rec == r); ix <- ix[order(tr_meta$start_ms[ix])]; n <- length(ix)
    fd[ix] <- floor(((seq_len(n) - 1 + round(shift * n)) %% n) / n * K) + 1
  }
  fd
}
## Both temperatures maximize the likelihood of the coupled three-class model.
coupled_nll <- function(u, sc, y) {
  L <- coupled_T(sc, u[1], u[2]); m <- apply(L, 1, max)
  -sum(L[cbind(seq_along(y), y)] - m - log(rowSums(exp(L - m))))
}
fit_T_cv <- function(K, shift) {
  fd <- block_folds(K, shift); sc <- matrix(NA, length(fd), 2, dimnames = list(NULL, c("AM", "SM")))
  for (f in 1:K) sc[fd == f, ] <- oof_scores(Xtr192[fd != f, ], ytr[fd != f], Xtr192[fd == f, ])
  optim(c(0, 0), coupled_nll, sc = sc, y = as.integer(ytr))$par
}
set.seed(2026)
n_runs <- 20
shifts <- c(0, runif(n_runs - 1))
cv_T <- do.call(rbind, lapply(c(10, 3), function(K) {
  u <- t(sapply(shifts, function(s) fit_T_cv(K, s)))
  data.frame(K = K, uAM = u[, 1], uSM = u[, 2],
             acc = mapply(test_acc_T, u[, 1], u[, 2]))
}))
acc_T10 <- cv_T$acc[cv_T$K == 10]; acc_T3 <- cv_T$acc[cv_T$K == 3]
d_T10 <- with(cv_T[cv_T$K == 10, ], uAM - uSM)
acc_lda_AS <- acc_drop["LDA", "drop13"]; acc_firth_AS <- acc_drop["Firth", "drop13"]

### Figure 11: Temperature scaling of the Firth partial model

Test accuracy of the Firth partial model with the pair $(A,S)$ dropped, when the log-odds of the classifiers $A$--$M$ and $S$--$M$ are divided by the temperatures $T_{AM}$ and $T_{SM}$. The cross marks the unscaled model and the dotted diagonal a single shared temperature. Circles and triangles show the temperatures chosen by 10-fold and 3-fold blocked cross-validation on the training clips, each in 20 runs with different block boundaries.

In [ ]:
options(repr.plot.width = 5.6, repr.plot.height = 4.3)

layout(matrix(1:2, 1), widths = c(4.7, 0.9))
par(mar = c(4.2, 4.4, 0.8, 0.6), mgp = c(2.5, 0.7, 0))
brk <- seq(floor(min(acc_T_grid) * 50) / 50, ceiling(max(acc_T_grid) * 50) / 50, by = 0.01)
pal <- hcl.colors(length(brk) - 1, "viridis")
image(u_grid, u_grid, acc_T_grid, breaks = brk, col = pal, useRaster = TRUE, las = 1,
      xlab = expression(log~T[AM]~~"(classifier A-M)"),
      ylab = expression(log~T[SM]~~"(classifier S-M)"))
contour(u_grid, u_grid, acc_T_grid, levels = c(0.7, 0.8, 0.9), add = TRUE,
        labcex = 0.7, col = "white", lwd = 0.8)
abline(0, 1, lty = 3, col = "white", lwd = 1.5)
points(0, 0, pch = 4, lwd = 2.5, col = "white", cex = 1.4)
text(0, 0, sprintf("no scaling (%.0f%%)", 100 * acc_firth_AS), pos = 2, col = "white", cex = 0.8)
with(cv_T[cv_T$K == 3, ], points(uAM, uSM, pch = 2, col = "#E69F00", lwd = 1.5, cex = 0.9))
with(cv_T[cv_T$K == 10, ], points(uAM, uSM, pch = 21, bg = "white", col = "black", cex = 1))
## Dark legend box, so that the white symbols are visible in it.
legend("topleft", bty = "o", bg = "grey20", box.col = NA, text.col = "white",
       cex = 0.75, inset = 0.02, seg.len = 2.2,
       legend = c("one shared temperature", "no scaling",
                  "10-fold CV (20 runs)", "3-fold CV (20 runs)"),
       lty = c(3, NA, NA, NA), lwd = c(1.5, NA, NA, NA),
       pch = c(NA, 4, 21, 2), pt.bg = c(NA, NA, "white", NA),
       col = c("white", "white", "black", "#E69F00"), pt.lwd = c(NA, 2.5, 1, 1.5))
par(mar = c(4.2, 0.4, 0.8, 3.4))
mids <- head(brk, -1) + 0.005
image(1, mids, matrix(mids, 1), breaks = brk, col = pal, xaxt = "n", yaxt = "n", xlab = "", ylab = "")
axis(4, at = pretty(brk), labels = paste0(round(100 * pretty(brk)), "%"), las = 1)
mtext("test accuracy", side = 4, line = 2.4, cex = 0.8)